# Loudness agreement: between lenses, and between signals

Every reasoning token of the **eval** set, one loudness plotted against another. Loudness is the mass
table's cell, `log P(any signal word)` over the whole vocabulary (`logprob_mass_full`), so the top-k
floor does not apply. All eight plots are drawn by the same two functions, `gather_pair` and
`plot_loudness_pair`.

**Part 1, lens vs lens** (same signal, x = J-lens, y = logit lens), at each signal's probe layer:

| model | signal | layer | eval set | trees |
|---|---|---|---|---|
| Qwen3.6-35B-A3B | grid | L27 | 52 trajectories | `qwen_p2_grid_selection_eval` |
| Qwen3.6-35B-A3B | direction | L27 | 52 trajectories | `qwen_p2_selection_eval` |
| gpt-oss-20b | grid | L14 | 720 (mass-era eval) | `gptoss_grid_mass_l14_eval` |
| gpt-oss-20b | direction | L15 | 720 (mass-era eval) | jlens: `mass_eval720_view`; logitlens: `logitlens_mass_l15` restricted to those 720 |

**Part 2, signal vs signal** (same lens, x = direction, y = grid), J-lens then logit lens, for each model.
Both signals are read at **one** layer, so the plot compares one residual stream against itself: L27 on
Qwen, and **L14** on gpt-oss, the only layer its grid tables hold (its direction tables cover L7-L23).
The direction and grid tables come from different gathers of the same trajectories; every token is
required to be present in both.

Every table is checked against its `.meta.json` vocabulary before it is read.

**Qwen caveat (2026-09-23).** Every Qwen lens output applied the final RMSNorm as `* w` instead of
`* (1 + w)`. Both Qwen lenses are affected, the logit lens more (its top-1 changes for 87% of L27 tokens
under the correct norm, against 19% for the J-lens): its loudness sits in a flat band near −6 for every
token. So the Qwen correlations, and the Qwen logit-lens plots above all, are **not** final. gpt-oss is
unaffected.

**Statistics.** Spearman ρ, with a 95% interval from resampling **trajectories** (tokens within one chain
are not independent), and Pearson r. The dashed line is y = x.

In [ ]:
# 1. imports and paths
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import pearsonr, spearmanr

from telos_interp.loudness_analysis.columns import LENS_LABEL, axis_label
from telos_interp.loudness_analysis.lens_io import check_vocabulary, read_mass_columns

REPO = Path("/workspace/repo/interp")
ACTS = Path("/workspace/activations")
FIG_DIR = Path("/workspace/results/lens_agreement")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The eight trees and vocabularies, named once.
QWEN_DIRECTION = ACTS / "qwen_p2_selection_eval"
QWEN_GRID = ACTS / "qwen_p2_grid_selection_eval"
GPTOSS_DIRECTION_JLENS = ACTS / "mass_eval720_view/activations"
GPTOSS_DIRECTION_LOGITLENS = ACTS / "logitlens_mass_l15"  # all 3,600; read only for the eval 720
GPTOSS_GRID = ACTS / "gptoss_grid_mass_l14_eval"
# The mass-era eval 720, by name: the one tree that holds only the eval set is where the list comes from,
# for a read whose own tree (the logit lens's 3,600) holds the train half as well.
GPTOSS_EVAL_720 = sorted(p.name for p in GPTOSS_GRID.glob("size*/*") if p.is_dir())

QWEN_DIRECTION_JSON = REPO / "data/jlens/qwen/direction_tokens_full_qwen3-6-35b-a3b.json"
QWEN_GRID_JSON = REPO / "data/jlens/qwen/grid_tokens_pruned_qwen3-6-35b-a3b.json"
GPTOSS_DIRECTION_JSON = REPO / "data/jlens/direction_tokens_full.json"
GPTOSS_GRID_JSON = REPO / "data/jlens/grid_tokens_pruned.json"

SUMMARY = []  # one row per plot, filled by plot_loudness_pair

In [ ]:
# 2. gather: one row per reasoning token, two loudnesses side by side
def read_loudness(tree, lens, layer, signal_json, names=None, threads=32):
    """One lens's signal loudness at one layer, for every reasoning token of a tree.

    Args:
        tree: holds `size*/<stem>/<stem>_<lens>_direction_mass.csv` (the file is called
            `direction_mass` whatever the signal; the sidecar says which vocabulary baked it)
        lens: "jlens" or "logitlens"
        layer: the `L{layer}` column to read
        signal_json: the vocabulary the tables must have been baked against, checked by file name
            against every `.meta.json` (the repo and deployed copies differ only in directory)
        names: read only these trajectories (default: every one in the tree)
        threads: parallel file reads (MooseFS is latency bound)

    Returns:
        DataFrame with columns name, size, step, abs_pos, value.
    """
    tree = Path(tree)
    paths = {p.parent.name: p for p in tree.glob(f"size*/*/*_{lens}_direction_mass.csv")}
    if names is not None:
        missing = sorted(set(names) - set(paths))
        if missing:
            raise FileNotFoundError(f"{len(missing)} trajectories have no {lens} table under {tree}, e.g. {missing[0]}")
        paths = {n: paths[n] for n in names}
    if not paths:
        raise FileNotFoundError(f"no {lens} mass tables under {tree}")

    vocab = check_vocabulary(list(paths.values()))  # raises on a mix
    if vocab is None or Path(vocab).name != Path(signal_json).name:
        raise ValueError(f"{tree} {lens} tables were baked against {vocab!r}, expected {signal_json}")

    def read_one(item):
        stem, path = item
        size = int(path.parent.parent.name.removeprefix("size"))
        table = read_mass_columns(path)
        rows = [(stem, size, step, pos, cells[layer]) for (step, pos), cells in table.items() if layer in cells]
        if len(rows) != len(table):
            raise ValueError(f"{stem}: {len(table) - len(rows)} tokens have no L{layer} in {path.name}")
        return rows

    with ThreadPoolExecutor(threads) as pool:
        rows = [r for chunk in pool.map(read_one, sorted(paths.items())) for r in chunk]
    return pd.DataFrame(rows, columns=["name", "size", "step", "abs_pos", "value"])


def gather_pair(x, y):
    """Join two loudnesses token by token. `x` and `y` are dicts of read_loudness's arguments.

    The trajectories are the ones x's tree holds (or x's `names`, when given); y's tree may hold more
    and is only read for those.
    Every token must be present in both, or this raises rather than joining a subset in silence.

    Returns:
        DataFrame with columns name, size, step, abs_pos, x, y.
    """
    dx = read_loudness(**x)
    dy = read_loudness(**y, names=sorted(dx.name.unique()))
    key = ["name", "step", "abs_pos"]
    df = dx.merge(dy.drop(columns="size"), on=key, how="outer", suffixes=("_x", "_y"), indicator=True)
    unmatched = int((df["_merge"] != "both").sum())
    if unmatched:
        raise ValueError(f"{unmatched} tokens are in only one of the two tables")
    df = df.rename(columns={"value_x": "x", "value_y": "y"}).drop(columns="_merge").reset_index(drop=True)
    print(f"{len(df):,} tokens from {df.name.nunique():,} trajectories")
    return df

In [ ]:
# 3. plot: token density of one loudness against another, with y = x
# One sequential ramp per model, light -> dark, so a Qwen plot and a gpt-oss plot never read as the same
# data. Blue is the palette's sequential hue; red is built around its categorical red, #e34948.
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
RED_RAMP = ["#fbd6d5", "#f5a9a8", "#ee7c7b", "#e34948", "#c0302f", "#951f1f", "#6b1414"]
DENSITY = {
    "Qwen3.6-35B-A3B": LinearSegmentedColormap.from_list("qwen", RED_RAMP),
    "gpt-oss-20b": LinearSegmentedColormap.from_list("gptoss", BLUE_RAMP),
}
INK, MUTED = "#1f1f1e", "#6b6b67"
MODEL_KEY = {"Qwen3.6-35B-A3B": "qwen", "gpt-oss-20b": "gptoss"}


def spearman_ci(df, n_boot=200, seed=0):
    """Spearman rho with a 95% interval from resampling trajectories, not tokens."""
    rho = spearmanr(df.x, df.y).statistic
    groups = [g.index.to_numpy() for _, g in df.groupby("name")]
    rng = np.random.default_rng(seed)
    x, y = df.x.to_numpy(), df.y.to_numpy()
    boots = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        boots.append(spearmanr(x[idx], y[idx]).statistic)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return rho, lo, hi


def plot_loudness_pair(df, model, x, y):
    """One plot: hexbin density of every token, y = x, and the two correlations.

    `x` and `y` are (lens, signal, layer) triples naming the two axes. Both axes share one range (the
    0.1-99.9 percentile of the two together), so the diagonal is the true y = x; the statistics use
    every token, including any outside the drawn range. Saved as PNG and PDF under FIG_DIR, and its
    numbers appended to SUMMARY.
    """
    (xl, xs, xn), (yl, ys, yn) = x, y
    rho, lo, hi = spearman_ci(df)
    r = pearsonr(df.x, df.y).statistic
    vmin, vmax = np.percentile(np.concatenate([df.x, df.y]), [0.1, 99.9])
    pad = 0.03 * (vmax - vmin)
    lim = (vmin - pad, vmax + pad)
    outside = int(((df.x < lim[0]) | (df.y < lim[0]) | (df.x > lim[1]) | (df.y > lim[1])).sum())

    # Name the plot by what it holds fixed: the signal for a lens pair, the lens for a signal pair.
    key = MODEL_KEY.get(model, model)
    if xs == ys and xn == yn:
        title, stem = f"{model} · {xs} · L{xn}", f"{key}_{xs}_L{xn}_{xl}_vs_{yl}"
    elif xl == yl and xn == yn:
        title, stem = f"{model} · {LENS_LABEL.get(xl, xl)} · L{xn}", f"{key}_{xl}_L{xn}_{xs}_vs_{ys}"
    else:
        title, stem = f"{model}", f"{key}_{xl}_{xs}_L{xn}_vs_{yl}_{ys}_L{yn}"

    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    hb = ax.hexbin(
        df.x, df.y, gridsize=70, extent=(*lim, *lim), bins="log", mincnt=1,
        cmap=DENSITY[model], linewidths=0,
    )
    ax.plot(lim, lim, ls="--", lw=1.2, color=MUTED, label="y = x")
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect("equal")
    ax.set_xlabel(axis_label(xl, xs, xn), color=INK)
    ax.set_ylabel(axis_label(yl, ys, yn), color=INK)
    ax.set_title(title, color=INK, loc="left")
    ax.text(
        0.03, 0.97,
        f"Spearman ρ = {rho:.3f}  [{lo:.3f}, {hi:.3f}]\nPearson r = {r:.3f}\n"
        f"{len(df):,} tokens · {df.name.nunique():,} trajectories"
        + (f"\n{outside:,} outside the drawn range" if outside else ""),
        transform=ax.transAxes, va="top", ha="left", fontsize=9, color=INK,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#d9d9d6", alpha=0.9),
    )
    ax.legend(loc="lower right", frameon=False, fontsize=9, labelcolor=MUTED)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#b5b5b0")
    ax.tick_params(colors=MUTED)
    cb = fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label("tokens per hexagon (log scale)", color=MUTED)
    cb.outline.set_visible(False)
    fig.tight_layout()

    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    plt.show()

    SUMMARY.append({
        "model": model, "x": f"{xl} {xs} L{xn}", "y": f"{yl} {ys} L{yn}",
        "tokens": len(df), "trajectories": df.name.nunique(),
        "spearman": rho, "spearman_lo": lo, "spearman_hi": hi, "pearson": r,
        "mean_x": df.x.mean(), "mean_y": df.y.mean(),
    })
    print(f"saved {FIG_DIR / stem}.{{png,pdf}}")

## Part 1: J-lens vs logit lens, same signal

In [ ]:
# 5. Qwen · grid: J-lens vs logit lens
qwen_grid = gather_pair(
    x=dict(tree=QWEN_GRID, lens="jlens", layer=27, signal_json=QWEN_GRID_JSON),
    y=dict(tree=QWEN_GRID, lens="logitlens", layer=27, signal_json=QWEN_GRID_JSON),
)
plot_loudness_pair(qwen_grid, model="Qwen3.6-35B-A3B", x=("jlens", "grid", 27), y=("logitlens", "grid", 27))

In [ ]:
# 6. Qwen · direction: J-lens vs logit lens
qwen_direction = gather_pair(
    x=dict(tree=QWEN_DIRECTION, lens="jlens", layer=27, signal_json=QWEN_DIRECTION_JSON),
    y=dict(tree=QWEN_DIRECTION, lens="logitlens", layer=27, signal_json=QWEN_DIRECTION_JSON),
)
plot_loudness_pair(qwen_direction, model="Qwen3.6-35B-A3B", x=("jlens", "direction", 27), y=("logitlens", "direction", 27))

In [ ]:
# 7. gpt-oss · grid: J-lens vs logit lens
gptoss_grid = gather_pair(
    x=dict(tree=GPTOSS_GRID, lens="jlens", layer=14, signal_json=GPTOSS_GRID_JSON),
    y=dict(tree=GPTOSS_GRID, lens="logitlens", layer=14, signal_json=GPTOSS_GRID_JSON),
)
plot_loudness_pair(gptoss_grid, model="gpt-oss-20b", x=("jlens", "grid", 14), y=("logitlens", "grid", 14))

In [ ]:
# 8. gpt-oss · direction: J-lens vs logit lens
gptoss_direction = gather_pair(
    x=dict(tree=GPTOSS_DIRECTION_JLENS, lens="jlens", layer=15, signal_json=GPTOSS_DIRECTION_JSON),
    y=dict(tree=GPTOSS_DIRECTION_LOGITLENS, lens="logitlens", layer=15, signal_json=GPTOSS_DIRECTION_JSON),
)
plot_loudness_pair(gptoss_direction, model="gpt-oss-20b", x=("jlens", "direction", 15), y=("logitlens", "direction", 15))

## Part 2: direction vs grid, same lens and layer

In [ ]:
# 9. Qwen · J-lens: direction vs grid
qwen_jlens_signals = gather_pair(
    x=dict(tree=QWEN_DIRECTION, lens="jlens", layer=27, signal_json=QWEN_DIRECTION_JSON),
    y=dict(tree=QWEN_GRID, lens="jlens", layer=27, signal_json=QWEN_GRID_JSON),
)
plot_loudness_pair(qwen_jlens_signals, model="Qwen3.6-35B-A3B", x=("jlens", "direction", 27), y=("jlens", "grid", 27))

In [ ]:
# 10. gpt-oss · J-lens: direction vs grid
gptoss_jlens_signals = gather_pair(
    x=dict(tree=GPTOSS_DIRECTION_JLENS, lens="jlens", layer=14, signal_json=GPTOSS_DIRECTION_JSON),
    y=dict(tree=GPTOSS_GRID, lens="jlens", layer=14, signal_json=GPTOSS_GRID_JSON),
)
plot_loudness_pair(gptoss_jlens_signals, model="gpt-oss-20b", x=("jlens", "direction", 14), y=("jlens", "grid", 14))

In [ ]:
# 11. Qwen · logit lens: direction vs grid
qwen_logitlens_signals = gather_pair(
    x=dict(tree=QWEN_DIRECTION, lens="logitlens", layer=27, signal_json=QWEN_DIRECTION_JSON),
    y=dict(tree=QWEN_GRID, lens="logitlens", layer=27, signal_json=QWEN_GRID_JSON),
)
plot_loudness_pair(qwen_logitlens_signals, model="Qwen3.6-35B-A3B", x=("logitlens", "direction", 27), y=("logitlens", "grid", 27))

In [ ]:
# 12. gpt-oss · logit lens: direction vs grid
gptoss_logitlens_signals = gather_pair(
    x=dict(tree=GPTOSS_DIRECTION_LOGITLENS, lens="logitlens", layer=14, signal_json=GPTOSS_DIRECTION_JSON, names=GPTOSS_EVAL_720),
    y=dict(tree=GPTOSS_GRID, lens="logitlens", layer=14, signal_json=GPTOSS_GRID_JSON),
)
plot_loudness_pair(gptoss_logitlens_signals, model="gpt-oss-20b", x=("logitlens", "direction", 14), y=("logitlens", "grid", 14))

In [ ]:
# 13. all eight plots side by side
summary = pd.DataFrame(SUMMARY)
summary.to_csv(FIG_DIR / "loudness_agreement_summary.csv", index=False)
summary.round(3)